# 06 · GAN condicional para generar granos de arroz sintéticos

**Tema 1 del proyecto (preparación de datos: GANs).** Una **GAN** (*Generative Adversarial
Network*) entrena dos redes que compiten:

- **Generador:** recibe ruido aleatorio y una clase (por ejemplo, "Basmati") y produce una imagen.
- **Discriminador:** recibe una imagen y su clase, y decide si es real o la inventó el generador.

El generador mejora a medida que engaña al discriminador. Es **condicional** (cGAN) porque se
le indica qué variedad dibujar, lo que permite generar imágenes etiquetadas para aumentar datos.

**Qué se hace y cómo se evalúa:**
1. Se entrena la cGAN a 64×64 píxeles **solo con el conjunto de train** (sin fuga de datos).
2. Se miran las imágenes generadas por clase junto a imágenes reales.
3. **¿Parecen de la variedad pedida?** Se las clasifica con el ResNet50V2 entrenado en el
   notebook 04.
4. **¿Sirven para aumentar datos?** Se entrena un clasificador pequeño con pocas imágenes
   reales, con y sin imágenes sintéticas, y se comparan los resultados.

**Requisitos de la sesión:** GPU T4 x2 (se usa una), Internet On y, como Input, el dataset,
`01-preparacion-datos` y `04-entrenamiento-final`.

**Salida:** `generador_cgan.keras`, `gan_historial.csv`, `gan_eval_clasificador.csv` y
`gan_aumento.csv`.

### Modo de prueba

Con `QUICK = True` todo corre en miniatura (1 época, pocas imágenes) para detectar errores en
pocos minutos. **Para la corrida real ponlo en `False` antes de Save & Run All.**

In [ ]:
QUICK = True

In [ ]:
import os
import time
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
SEED = 42
IMG = 64          # resolución de la GAN
LATENT = 128      # tamaño del vector de ruido
keras.utils.set_random_seed(SEED)


def find_dir_with(*names, base="/kaggle/input"):
    for root, _, files in os.walk(base):
        if all(n in files for n in names):
            return Path(root)
    raise FileNotFoundError(f"No se encontró {names}: revisa los Input del notebook")


SPLITS = find_dir_with("train.csv", "rice_data.py")
FINAL = find_dir_with("modelo_resnet50v2.keras")
train_df = pd.read_csv(SPLITS / "train.csv")
val_df = pd.read_csv(SPLITS / "val.csv")
CLASSES = train_df[["label_idx", "label"]].drop_duplicates().sort_values("label_idx")["label"].tolist()
N_CLASES = len(CLASSES)

DATA_ROOT = None
for root, dirs, _ in os.walk("/kaggle/input"):
    if set(CLASSES) <= set(dirs):
        DATA_ROOT = Path(root)
        break

EPOCHS_GAN = 1 if QUICK else 30
if QUICK:
    train_df = train_df.groupby("label_idx", group_keys=False).sample(n=200, random_state=SEED)
print("GPU:", tf.config.list_physical_devices("GPU"))
print(f"Imágenes de train para la GAN: {len(train_df)}  |  épocas: {EPOCHS_GAN}")

## 1. Datos a 64×64

Las imágenes se reducen de 250×250 a 64×64 para que la GAN entrene en un tiempo razonable, y
se escalan a [-1, 1], el rango de salida del generador (activación `tanh`). Se guardan en
memoria como enteros de 8 bits (cerca de 640 MB) para no releer el disco en cada época.

In [ ]:
def leer(path, label):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.resize(img, (IMG, IMG), antialias=True)
    return tf.cast(tf.round(img), tf.uint8), label


def rutas(df):
    return (str(DATA_ROOT) + "/" + df["filepath"]).to_numpy(), df["label_idx"].to_numpy()


BATCH = 128
gan_ds = (tf.data.Dataset.from_tensor_slices(rutas(train_df))
          .map(leer, num_parallel_calls=tf.data.AUTOTUNE)
          .cache()
          .shuffle(len(train_df), seed=SEED)
          .batch(BATCH, drop_remainder=True)
          .map(lambda x, y: (tf.cast(x, tf.float32) / 127.5 - 1.0, y), num_parallel_calls=tf.data.AUTOTUNE)
          .prefetch(tf.data.AUTOTUNE))


def cargar_arreglo(df):
    """Carga imágenes a 64×64 como arreglo numpy en [0, 255] (para evaluar)."""
    ds = tf.data.Dataset.from_tensor_slices(rutas(df)).map(leer, num_parallel_calls=tf.data.AUTOTUNE).batch(256)
    xs, ys = zip(*[(x.numpy(), y.numpy()) for x, y in ds])
    return np.concatenate(xs).astype("float32"), np.concatenate(ys)

## 2. Generador y discriminador

- **Generador:** ruido (128) + clase → capa densa a 4×4×512 → cuatro convoluciones transpuestas
  que duplican el tamaño (8, 16, 32, 64) → imagen 64×64×3.
- **Discriminador:** imagen + un canal extra que codifica la clase → cuatro convoluciones con
  paso 2 → probabilidad de que la imagen sea real.

Es la arquitectura DCGAN, con BatchNormalization en el generador y LeakyReLU y Dropout en el
discriminador para estabilizar el entrenamiento.

In [ ]:
def construir_generador():
    z = keras.Input((LATENT,))
    c = keras.Input((1,), dtype="int32")
    emb = layers.Flatten()(layers.Embedding(N_CLASES, 50)(c))
    x = layers.Concatenate()([z, emb])
    x = layers.Dense(4 * 4 * 512, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Reshape((4, 4, 512))(x)
    for filtros in (256, 128, 64):
        x = layers.Conv2DTranspose(filtros, 4, strides=2, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.LeakyReLU(0.2)(x)
    img = layers.Conv2DTranspose(3, 4, strides=2, padding="same", activation="tanh")(x)
    return keras.Model([z, c], img, name="generador")


def construir_discriminador():
    img = keras.Input((IMG, IMG, 3))
    c = keras.Input((1,), dtype="int32")
    mapa = layers.Reshape((IMG, IMG, 1))(layers.Dense(IMG * IMG)(layers.Flatten()(layers.Embedding(N_CLASES, 50)(c))))
    x = layers.Concatenate()([img, mapa])
    for filtros in (64, 128, 256, 512):
        x = layers.Conv2D(filtros, 4, strides=2, padding="same")(x)
        x = layers.LeakyReLU(0.2)(x)
        x = layers.Dropout(0.3)(x)
    x = layers.Flatten()(x)
    return keras.Model([img, c], layers.Dense(1)(x), name="discriminador")


generador, discriminador = construir_generador(), construir_discriminador()
print(f"Generador: {generador.count_params() / 1e6:.2f} M parámetros")
print(f"Discriminador: {discriminador.count_params() / 1e6:.2f} M parámetros")

## 3. Entrenamiento adversarial

En cada paso:
1. **Discriminador:** ve imágenes reales (objetivo 0,9: *label smoothing*, que evita que se
   vuelva demasiado seguro) e imágenes generadas (objetivo 0) y aprende a separarlas.
2. **Generador:** genera imágenes nuevas y aprende a que el discriminador las marque como reales.

Optimizador Adam con tasa 0,0002 y β₁ = 0,5, los valores estándar de DCGAN.

In [ ]:
class CGAN(keras.Model):
    def __init__(self, generador, discriminador):
        super().__init__()
        self.generador, self.discriminador = generador, discriminador
        self.bce = keras.losses.BinaryCrossentropy(from_logits=True)

    def compile(self, d_opt, g_opt):
        super().compile()
        self.d_opt, self.g_opt = d_opt, g_opt

    def train_step(self, data):
        reales, clases = data
        clases = tf.reshape(clases, (-1, 1))
        n = tf.shape(reales)[0]

        z = tf.random.normal((n, LATENT))
        falsas = self.generador([z, clases], training=True)
        with tf.GradientTape() as tape:
            d_real = self.discriminador([reales, clases], training=True)
            d_falso = self.discriminador([falsas, clases], training=True)
            d_loss = self.bce(tf.ones_like(d_real) * 0.9, d_real) + self.bce(tf.zeros_like(d_falso), d_falso)
        grads = tape.gradient(d_loss, self.discriminador.trainable_variables)
        self.d_opt.apply_gradients(zip(grads, self.discriminador.trainable_variables))

        z = tf.random.normal((n, LATENT))
        with tf.GradientTape() as tape:
            d_falso = self.discriminador([self.generador([z, clases], training=True), clases], training=True)
            g_loss = self.bce(tf.ones_like(d_falso), d_falso)
        grads = tape.gradient(g_loss, self.generador.trainable_variables)
        self.g_opt.apply_gradients(zip(grads, self.generador.trainable_variables))
        return {"d_loss": d_loss, "g_loss": g_loss}


def a_pixeles(x):
    return np.clip((np.asarray(x) + 1.0) * 127.5, 0, 255).astype("uint8")


# Ruido fijo para ver la evolución de las mismas muestras a lo largo del entrenamiento.
Z_FIJO = tf.random.normal((N_CLASES * 4, LATENT), seed=SEED)
C_FIJO = tf.constant(np.repeat(np.arange(N_CLASES), 4).reshape(-1, 1), dtype=tf.int32)
instantaneas = {}


class Instantanea(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        e = epoch + 1
        if e == 1 or e % 5 == 0 or e == EPOCHS_GAN:
            instantaneas[e] = a_pixeles(generador([Z_FIJO, C_FIJO], training=False))


gan = CGAN(generador, discriminador)
gan.compile(d_opt=keras.optimizers.Adam(2e-4, beta_1=0.5), g_opt=keras.optimizers.Adam(2e-4, beta_1=0.5))
t0 = time.time()
hist = gan.fit(gan_ds, epochs=EPOCHS_GAN, callbacks=[Instantanea()], verbose=2)
print(f"GAN entrenada en {(time.time() - t0) / 60:.1f} min")

generador.save(OUT_DIR / "generador_cgan.keras")
pd.DataFrame(hist.history).to_csv(OUT_DIR / "gan_historial.csv", index_label="epoch")

## 4. Curvas de pérdida y evolución del generador

En una GAN las pérdidas **no deben bajar a cero**: si el discriminador gana por completo, el
generador deja de aprender. Lo deseable es que ambas se mantengan en equilibrio, oscilando.

In [ ]:
h = pd.DataFrame(hist.history)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(np.arange(1, len(h) + 1), h["d_loss"], marker="o", label="discriminador")
ax.plot(np.arange(1, len(h) + 1), h["g_loss"], marker="o", label="generador")
ax.set_xlabel("Época")
ax.set_ylabel("Pérdida")
ax.set_title("Pérdidas de la GAN")
ax.legend()
plt.tight_layout()
plt.show()

epocas = sorted(instantaneas)
fig, axes = plt.subplots(len(epocas), N_CLASES, figsize=(2 * N_CLASES, 2.1 * len(epocas)), squeeze=False)
for fila, e in enumerate(epocas):
    for c in range(N_CLASES):
        ax = axes[fila, c]
        ax.imshow(instantaneas[e][c * 4])
        ax.set_xticks([]), ax.set_yticks([])
        if fila == 0:
            ax.set_title(CLASSES[c])
        if c == 0:
            ax.set_ylabel(f"época {e}")
plt.suptitle("La misma muestra de ruido a lo largo del entrenamiento")
plt.tight_layout()
plt.show()

## 5. Imágenes reales frente a generadas

In [ ]:
N_MUESTRA = 6
fig, axes = plt.subplots(N_CLASES, 2 * N_MUESTRA, figsize=(1.6 * 2 * N_MUESTRA, 1.8 * N_CLASES))
for c in range(N_CLASES):
    reales = train_df[train_df["label_idx"] == c].sample(N_MUESTRA, random_state=SEED)
    x_real, _ = cargar_arreglo(reales)
    z = tf.random.normal((N_MUESTRA, LATENT), seed=SEED + c)
    x_gen = a_pixeles(generador([z, tf.fill((N_MUESTRA, 1), c)], training=False))
    for j in range(N_MUESTRA):
        axes[c, j].imshow(x_real[j].astype("uint8"))
        axes[c, N_MUESTRA + j].imshow(x_gen[j])
    axes[c, 0].set_ylabel(CLASSES[c], rotation=0, ha="right", va="center")
for ax in axes.ravel():
    ax.set_xticks([]), ax.set_yticks([])
axes[0, N_MUESTRA // 2].set_title("Reales (64×64)")
axes[0, N_MUESTRA + N_MUESTRA // 2].set_title("Generadas por la cGAN")
plt.tight_layout()
plt.show()

## 6. ¿Las imágenes generadas parecen de la variedad pedida?

Se generan imágenes de cada clase y se clasifican con el **ResNet50V2 entrenado en el notebook
04** (99,91 % en test). Si las clasifica en la variedad que se le pidió a la GAN, el generador
aprendió los rasgos de cada variedad.

Comparación justa: las imágenes generadas son de 64×64 y ResNet espera 224×224, así que se
amplían. Como la pérdida de resolución por sí sola puede confundir al clasificador, también
se evalúan **imágenes reales reducidas a 64×64 y ampliadas igual** como referencia.

In [ ]:
clasificador = keras.models.load_model(FINAL / "modelo_resnet50v2.keras", compile=False)


def predecir_224(x_0_255):
    x = tf.image.resize(tf.convert_to_tensor(x_0_255, tf.float32), (224, 224))
    return clasificador.predict(x, batch_size=64, verbose=0).argmax(1)


N_EVAL = 40 if QUICK else 400
val_eval = val_df.groupby("label_idx", group_keys=False).sample(n=N_EVAL, random_state=SEED)
x_val64, y_val64 = cargar_arreglo(val_eval)

x_gen, y_gen = [], []
for c in range(N_CLASES):
    z = tf.random.normal((N_EVAL, LATENT), seed=SEED + 100 + c)
    x_gen.append(a_pixeles(generador([z, tf.fill((N_EVAL, 1), c)], training=False)).astype("float32"))
    y_gen.append(np.full(N_EVAL, c))
x_gen, y_gen = np.concatenate(x_gen), np.concatenate(y_gen)

pred_real, pred_gen = predecir_224(x_val64), predecir_224(x_gen)
eval_clf = pd.DataFrame({
    "clase": CLASSES + ["TOTAL"],
    "reales 64×64 (referencia)": [np.mean(pred_real[y_val64 == c] == c) for c in range(N_CLASES)] + [np.mean(pred_real == y_val64)],
    "generadas por la cGAN": [np.mean(pred_gen[y_gen == c] == c) for c in range(N_CLASES)] + [np.mean(pred_gen == y_gen)],
})
eval_clf.to_csv(OUT_DIR / "gan_eval_clasificador.csv", index=False)
eval_clf.round(3)

## 7. ¿Las imágenes sintéticas sirven para aumentar datos?

Experimento con **pocos datos reales**, que es donde una GAN podría ayudar:

| Escenario | Datos de entrenamiento por clase |
|---|---|
| A · pocos reales | 50 reales |
| B · pocos reales + sintéticas | 50 reales + 200 generadas por la cGAN |
| C · más reales (referencia) | 250 reales |

Se entrena el mismo clasificador pequeño (64×64) en los tres escenarios, con **3 semillas**
distintas, y se evalúa en las mismas imágenes de validación.

**Nota metodológica.** Cada escenario recibe el **mismo número de pasos de entrenamiento**
(1.000 actualizaciones de pesos), no el mismo número de épocas. Con épocas iguales, el
escenario A (250 imágenes) haría 5 veces menos actualizaciones que B y C, y la comparación
mediría eso y no el valor de los datos. El clasificador además **no usa BatchNormalization**:
en una primera corrida, con épocas iguales y BatchNorm, A quedó en 20 % exacto (predecía una sola
clase) porque con tan pocos pasos las estadísticas de BatchNorm no alcanzaban a estabilizarse.
Era un defecto del montaje, no falta de información en los datos. Si B supera a A, las imágenes
sintéticas aportan información útil. C indica cuánto se ganaría consiguiendo datos reales.

Ojo: la GAN se entrenó con todo el train, así que en B el generador "conoce" más de 50
imágenes por clase. El experimento mide si ese conocimiento se puede transferir a otro
modelo a través de imágenes sintéticas, que es justamente el uso práctico de una GAN para
aumentar datos.

In [ ]:
def clasificador_64():
    inp = keras.Input((IMG, IMG, 3))
    x = layers.Rescaling(1 / 127.5, offset=-1)(inp)
    for filtros in (32, 64, 128):
        x = layers.Conv2D(filtros, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filtros, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    m = keras.Model(inp, layers.Dense(N_CLASES, activation="softmax")(x))
    m.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


N_POCOS, N_SINT, N_MAS = (10, 20, 30) if QUICK else (50, 200, 250)
PASOS = 10 if QUICK else 1000   # mismas actualizaciones de pesos en los tres escenarios
LOTE = 64
filas_a = []
for semilla in (0, 1, 2):
    keras.utils.set_random_seed(semilla)
    pool = train_df.groupby("label_idx", group_keys=False).sample(n=N_MAS, random_state=semilla)
    x_mas, y_mas = cargar_arreglo(pool)
    pocos = np.concatenate([np.where(y_mas == c)[0][:N_POCOS] for c in range(N_CLASES)])
    x_pocos, y_pocos = x_mas[pocos], y_mas[pocos]

    x_s, y_s = [], []
    for c in range(N_CLASES):
        z = tf.random.normal((N_SINT, LATENT), seed=1000 * semilla + c)
        x_s.append(a_pixeles(generador([z, tf.fill((N_SINT, 1), c)], training=False)).astype("float32"))
        y_s.append(np.full(N_SINT, c))
    x_aum, y_aum = np.concatenate([x_pocos, *x_s]), np.concatenate([y_pocos, *y_s])

    for escenario, (x, y) in {"A · pocos reales": (x_pocos, y_pocos),
                              "B · pocos reales + sintéticas": (x_aum, y_aum),
                              "C · más reales (referencia)": (x_mas, y_mas)}.items():
        modelo = clasificador_64()
        epocas = max(1, round(PASOS * LOTE / len(y)))
        modelo.fit(x, y, epochs=epocas, batch_size=LOTE, shuffle=True, verbose=0)
        acc = modelo.evaluate(x_val64, y_val64, verbose=0)[1]
        filas_a.append({"escenario": escenario, "semilla": semilla, "imágenes de entrenamiento": len(y),
                        "épocas": epocas, "accuracy val": acc})
        print(f"semilla {semilla}  {escenario:32s} épocas={epocas:4d}  acc={acc:.4f}")

aumento = pd.DataFrame(filas_a)
aumento.to_csv(OUT_DIR / "gan_aumento.csv", index=False)
aumento.groupby("escenario")["accuracy val"].agg(["mean", "std"]).round(4)

**Cómo leer los resultados:**

- **Sección 6:** si la exactitud sobre las imágenes generadas se acerca a la referencia de
  imágenes reales a 64×64, la cGAN reproduce bien los rasgos de cada variedad. Las clases donde
  baja más son las que el generador aún no domina.
- **Sección 7:** compara A con B. Si B es mayor, las sintéticas sirven como aumento de datos.
  Si son parecidas, la GAN no aporta información nueva a este clasificador. Si B es menor, las
  imágenes generadas tienen defectos que confunden al modelo.
- **Contexto:** este dataset ya tiene 15.000 imágenes balanceadas por clase, así que en la
  práctica no necesita aumento con GAN. Aquí la GAN se presenta como técnica de preparación de
  datos para escenarios con pocas imágenes, y el experimento A/B/C muestra su utilidad real.